# 00 — Data Loading & Feature Engineering
**Responsável**: R1 (Remoto)

**Objetivo**: Carregar os CSVs brutos, calcular features por artista×país e salvar `.parquet`.

> ⚠️ O charts_songs_daily.csv tem ~42M linhas (~10GB). Usamos **Dask**.

In [2]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PILOT_COUNTRIES = ['br', 'us', 'gb', 'mx', 'ar']
print('Setup OK')

Setup OK


## 1. Carregar artists.csv

In [3]:
df_artists = pd.read_csv('../artists.csv')
print(f'Artists: {df_artists.shape}')
print('Missing:', df_artists.isnull().sum().to_dict())
df_artists.head()

Artists: (72982, 6)
Missing: {'artist_uri': 0, 'artist_name': 3, 'monthly_listeners': 48037, 'monthly_listeners_rank': 48037, 'monthly_listeners_peak_rank': 70482, 'monthly_listeners_peak_listeners': 48037}


,artist_uri,artist_name,monthly_listeners,monthly_listeners_rank,monthly_listeners_peak_rank,monthly_listeners_peak_listeners
0,spotify:artist:1UwTwoC4T1i6vzwsQgIWB0,Feza,1642648.0,9287.0,NaN,1781914.0
1,spotify:artist:3TVXtAsR1Inumwj472S9r4,Drake,91729711.0,12.0,3.0,100057064.0
2,spotify:artist:2zMD4U9OQAR3xuq6cjer8p,Calvin Fallo,972768.0,14746.0,NaN,972768.0
3,spotify:artist:2W0Thu5zSRJV1cW2R50UPU,Credo V Daniels,NaN,NaN,NaN,NaN
4,spotify:artist:0iy6SpAZnBvSYpBw2bFoUQ,PLG Chanty,967892.0,14820.0,NaN,979004.0


## 2. Carregar charts com Dask + filtrar países

In [4]:
dtypes = {
    'date':'str','country':'str','rank':'int64','uri':'str',
    'artist_names':'str','track_name':'str','label':'str',
    'peak_rank':'float64','previous_rank':'float64',
    'days_on_chart':'float64','streams':'float64',
    'consecutive_days':'float64','entry_status':'str',
    'peak_date':'str','entry_rank':'float64','entry_date':'str',
    'release_date':'str','artist_uris':'str'
}
ddf = dd.read_csv('../charts_songs_daily.csv', dtype=dtypes, on_bad_lines='skip')
print(f'Partições: {ddf.npartitions}')

Partições: 164


In [11]:
print(f'Filtrando países: {PILOT_COUNTRIES}')
df_charts = ddf[ddf['country'].isin(PILOT_COUNTRIES)].compute()
for col in ['date','peak_date','entry_date','release_date']:
    df_charts[col] = pd.to_datetime(df_charts[col], errors='coerce')
print(f'Resultado: {df_charts.shape}')
print(df_charts['country'].value_counts())

Filtrando países: ['br', 'us', 'gb', 'mx', 'ar']
Resultado: (3434995, 18)
country
ar    687000
br    687000
mx    687000
gb    686998
us    686997
Name: count, dtype: int64[pyarrow]


## 3. Explodir artist_uris (collabs)

In [12]:
df_charts['artist_uri_list'] = df_charts['artist_uris'].str.split('|')
df_charts['artist_name_list'] = df_charts['artist_names'].str.split('|')
df_exp = df_charts.explode(['artist_uri_list', 'artist_name_list'])
df_exp = df_exp.rename(columns={'artist_uri_list':'artist_uri', 'artist_name_list':'artist_name_solo'})
df_exp['artist_uri'] = df_exp['artist_uri'].str.strip()
df_exp['artist_name_solo'] = df_exp['artist_name_solo'].str.strip()
print(f'Antes: {len(df_charts)} → Depois: {len(df_exp)}')

Antes: 3434995 → Depois: 5805507


## 4. Feature Engineering

In [14]:
ref_date = df_exp['date'].max()
grouped = df_exp.groupby(['artist_uri','country'])

features = grouped.agg(
    artist_name=('artist_name_solo','first'),
    total_tracks=('uri','nunique'),
    total_streams=('streams','sum'),
    avg_rank=('rank','mean'),
    best_rank=('rank','min'),
    days_on_chart_total=('days_on_chart','max'),
    first_entry_date=('date','min'),
    last_seen_date=('date','max'),
    entry_count=('entry_status', lambda x: (x=='NEW_ENTRY').sum()),
    label_mode=('label', lambda x: x.mode().iloc[0] if len(x.mode())>0 else 'Unknown'),
).reset_index()

# Stream concentration
ts = df_exp.groupby(['artist_uri','country','uri'])['streams'].sum().reset_index()
top_t = ts.sort_values('streams',ascending=False).groupby(['artist_uri','country']).first().reset_index()
tot = ts.groupby(['artist_uri','country'])['streams'].sum().reset_index(name='tot_s')
conc = top_t.merge(tot, on=['artist_uri','country'])
conc['stream_concentration'] = conc['streams']/conc['tot_s']
features = features.merge(conc[['artist_uri','country','stream_concentration']], on=['artist_uri','country'], how='left')

# Trends 30d / 90d
for d, col in [(30,'trend_30d'),(90,'trend_90d')]:
    cut = ref_date - pd.Timedelta(days=d)
    r = df_exp[df_exp['date']>=cut].groupby(['artist_uri','country'])['streams'].sum().reset_index(name='s_r')
    o = df_exp[(df_exp['date']>=cut-pd.Timedelta(days=d))&(df_exp['date']<cut)].groupby(['artist_uri','country'])['streams'].sum().reset_index(name='s_o')
    t = r.merge(o, on=['artist_uri','country'], how='outer').fillna(0)
    t[col] = np.where(t['s_o']>0, (t['s_r']-t['s_o'])/t['s_o'], np.where(t['s_r']>0,1.0,0.0))
    features = features.merge(t[['artist_uri','country',col]], on=['artist_uri','country'], how='left')

features['days_since_last_seen'] = (ref_date - features['last_seen_date']).dt.days
print(f'Features: {features.shape}')

Features: (13220, 16)


## 5. Join com artists.csv

In [15]:
ai = df_artists[['artist_uri','monthly_listeners','monthly_listeners_peak_listeners']].copy()
ai.columns = ['artist_uri','monthly_listeners','peak_listeners']
features = features.merge(ai, on='artist_uri', how='left')
features['listener_ratio'] = np.where(features['peak_listeners']>0, features['monthly_listeners']/features['peak_listeners'], np.nan)
print(features.shape)

(13220, 19)


## 5.1 Consolidar artist_uri duplicados (mesmo artista, perfis diferentes)

Ao investigar por que artistas consagrados apareciam mal posicionados no modelo (ex. Henrique & Juliano com `avg_rank` pior do que o esperado), descobrimos que o mesmo artista às vezes tem **mais de um `artist_uri`** no Spotify (perfil legado/duplicado) — o pipeline de agregação trata cada `artist_uri` como uma entidade separada, então o sinal do artista principal (streams, dias no chart, faixas) fica fragmentado entre dois perfis.

Consolidamos por `(artist_name, country)` quando há mais de um `artist_uri` para o mesmo nome:
- `artist_name` vazio/nulo **nunca** é agrupado (evita juntar artistas diferentes por acidente de nome faltante);
- `total_tracks`, `total_streams`, `entry_count` somam entre os perfis;
- `avg_rank` fica ponderado por `days_on_chart_total` de cada perfil (perfil com mais presença no chart pesa mais na média);
- `best_rank` = mínimo, datas = min/max entre os perfis;
- o `artist_uri` "canônico" salvo é o do perfil com mais streams (o "principal").

In [ ]:
print('=== ANTES da consolidação ===')
print(f'Total de linhas: {len(features)}')

valid_name = features['artist_name'].notna() & (features['artist_name'].str.strip() != '')
dup_key = features.loc[valid_name].groupby(['artist_name', 'country'])['artist_uri'].transform('nunique')
dup_rows = features.loc[valid_name][dup_key > 1]
print(f'Artistas com mais de um artist_uri no mesmo país: {dup_rows.groupby(["artist_name","country"]).ngroups} grupos ({len(dup_rows)} linhas)')

example_before = features[features['artist_name'] == 'Henrique & Juliano']
print('\nExemplo -- Henrique & Juliano ANTES da consolidação:')
print(example_before[['artist_uri', 'country', 'total_tracks', 'total_streams', 'avg_rank']].to_string(index=False))

# --- consolidação ---
singles = features[~valid_name | (valid_name & (dup_key == 1))].copy()
dupes = features[valid_name & (dup_key > 1)].copy()

def consolidate_group(name, country, g):
    g = g.sort_values('total_streams', ascending=False)
    dominant = g.iloc[0]
    total_streams = g['total_streams'].sum()
    weights = g['days_on_chart_total'].clip(lower=1)
    avg_rank_w = (g['avg_rank'] * weights).sum() / weights.sum()
    concentration_w = (
        (g['stream_concentration'] * g['total_streams']).sum() / total_streams
        if total_streams > 0 else g['stream_concentration'].mean()
    )
    monthly_listeners = g['monthly_listeners'].dropna()
    peak_listeners = g['peak_listeners'].dropna()
    return pd.Series({
        'artist_uri': dominant['artist_uri'], 'country': country, 'artist_name': name,
        'total_tracks': g['total_tracks'].sum(), 'total_streams': total_streams,
        'avg_rank': avg_rank_w, 'best_rank': g['best_rank'].min(),
        'days_on_chart_total': g['days_on_chart_total'].max(),
        'first_entry_date': g['first_entry_date'].min(), 'last_seen_date': g['last_seen_date'].max(),
        'entry_count': g['entry_count'].sum(), 'label_mode': dominant['label_mode'],
        'stream_concentration': concentration_w,
        'trend_30d': dominant['trend_30d'], 'trend_90d': dominant['trend_90d'],
        'monthly_listeners': monthly_listeners.max() if len(monthly_listeners) else np.nan,
        'peak_listeners': peak_listeners.max() if len(peak_listeners) else np.nan,
        'merged_uris_count': len(g),
    })

rows = [consolidate_group(name, country, g) for (name, country), g in dupes.groupby(['artist_name', 'country'])]
consolidated = pd.DataFrame(rows).reset_index(drop=True)
consolidated['listener_ratio'] = np.where(
    consolidated['peak_listeners'] > 0, consolidated['monthly_listeners'] / consolidated['peak_listeners'], np.nan
)
singles['merged_uris_count'] = 1

ref_date = pd.concat([singles['last_seen_date'], consolidated['last_seen_date']]).max()
features = pd.concat([singles, consolidated], ignore_index=True, sort=False)
features['days_since_last_seen'] = (ref_date - features['last_seen_date']).dt.days
features['merged_uris_count'] = features['merged_uris_count'].fillna(1).astype(int)

print('\n=== DEPOIS da consolidação ===')
print(f'Total de linhas: {len(features)} (eram {len(singles) + len(dup_rows)})')
print(f'Artistas consolidados (merged_uris_count > 1): {(features["merged_uris_count"] > 1).sum()}')

example_after = features[features['artist_name'] == 'Henrique & Juliano']
print('\nExemplo -- Henrique & Juliano DEPOIS da consolidação:')
print(example_after[['artist_uri', 'country', 'total_tracks', 'total_streams', 'avg_rank', 'merged_uris_count']].to_string(index=False))

## 6. Salvar

In [16]:
features.to_parquet(PROCESSED_DIR/'artist_country_features.parquet', index=False)
sample = features[features['country']=='br'].nlargest(100,'total_streams')
sample.to_csv(PROCESSED_DIR/'sample_br_100.csv', index=False)
print(f'✅ Parquet salvo. Amostra BR: {len(sample)} artistas')
sample[['artist_name','total_streams','total_tracks','avg_rank','stream_concentration','listener_ratio']].head(10)

✅ Parquet salvo. Amostra BR: 100 artistas


,artist_name,total_streams,total_tracks,avg_rank,stream_concentration,listener_ratio
6478,Henrique & Juliano,7.213092e+09,212,97.819450,0.042702,0.846691
3327,Marília Mendonça,5.288258e+09,171,86.426410,0.044468,0.793583
12406,Gusttavo Lima,4.589981e+09,204,84.172691,0.036796,0.979935
6964,Zé Neto & Cristiano,4.529850e+09,125,84.609811,0.069003,0.986541
3763,Ana Castela,4.431963e+09,100,79.898536,0.073503,0.574158
2809,Jorge & Mateus,4.389975e+09,125,88.535220,0.049665,0.853598
11957,MC Ryan SP,4.369129e+09,152,96.265565,0.069931,0.864385
4389,Matheus & Kauan,4.113411e+09,142,84.419006,0.053435,0.893755
8724,Maiara & Maraisa,3.016666e+09,121,82.980423,0.077953,0.677197
11685,Grupo Menos É Mais,2.913391e+09,56,93.424321,0.165764,0.665459
